<div style="font-family: 'Helvetica Neue', -apple-system, Arial, sans-serif;">

### Playground to Test: Do Signals have Predictive Power?
___

</div>

In [1]:
%load_ext autoreload
%autoreload 2

import sys
from pathlib import Path
sys.path.append(str(Path().resolve().parent))

from modules.signals.fetch import get_sp500_universe, fetch_price_data, START_DATE, END_DATE
from modules.signals.momentum import compute_momentum
from modules.signals.mean_reversion import compute_mean_reversion
from modules.signals.testing import evaluate_signal

import pandas as pd

In [ ]:
from modules.signals.fetch import save_price_data

universe = get_sp500_universe()
print(f"{len(universe)} tickers across {universe['Sector'].nunique()} sectors")
close, volume = fetch_price_data(universe["Symbol"].tolist(), start=START_DATE, end=END_DATE)

failed_tickers = close.columns[close.isna().all()].tolist()
if failed_tickers:
    close = close.drop(columns=failed_tickers)
    volume = volume.drop(columns=failed_tickers)

save_price_data(universe, close, volume)
print(f"Close shape: {close.shape}")

# once the data's been saved...

# from modules.signals.fetch import load_price_data

# universe, close, volume = load_price_data()
# print(f"Close shape: {close.shape}")
# close.tail()

Fetching batch 1 / 13 (40 tickers)...


[*********************100%***********************]  40 of 40 completed


Fetching batch 2 / 13 (40 tickers)...


[*********************100%***********************]  40 of 40 completed


Fetching batch 3 / 13 (40 tickers)...


...

Fetching batch 12 / 13 (40 tickers)...


[*********************100%***********************]  40 of 40 completed


Fetching batch 13 / 13 (23 tickers)...


[*********************100%***********************]  23 of 23 completed


Saved universe, close, and volume to /Users/paulgrajzl/Documents/Quant-Projects/systematic-portfolio-engine-equities-1/data/
Close shape: (1907, 503)


In [5]:
momentum_signal = compute_momentum(close)
mean_reversion_signal = compute_mean_reversion(close)

print(f"Momentum shape: {momentum_signal.shape}, non-null: {momentum_signal.notna().sum().sum()}")
print(f"Mean Reversion shape: {mean_reversion_signal.shape}, non-null: {mean_reversion_signal.notna().sum().sum()}")

Momentum shape: (1907, 503), non-null: 907326
Mean Reversion shape: (1907, 503), non-null: 936240


In [6]:
HORIZON = 20  # arbitrary starting point, just to confirm the pipeline runs

mom_ic_series, mom_summary = evaluate_signal(momentum_signal, close, horizon=HORIZON)
print(pd.Series(mom_summary).round(4))

Mean IC                -0.0147
IC Std                  0.1954
Information Ratio      -0.0754
% Positive IC           0.5126
N Observations       1824.0000
dtype: float64


In [7]:
mr_ic_series, mr_summary = evaluate_signal(mean_reversion_signal, close, horizon=HORIZON)
print(pd.Series(mr_summary).round(4))

Mean IC                 0.0077
IC Std                  0.1766
Information Ratio       0.0437
% Positive IC           0.5138
N Observations       1882.0000
dtype: float64


In [8]:
from modules.standardization.zscore import zscore_signal, winsorize_signal, standardize_signal

In [9]:
momentum_standardized = standardize_signal(momentum_signal)
mean_reversion_standardized = standardize_signal(mean_reversion_signal)

print("Momentum standardized:")
print(momentum_standardized.iloc[-1].describe())

Momentum standardized:
count    501.000000
mean      -0.022428
std        0.906576
min       -2.509160
25%       -0.625165
50%       -0.068416
75%        0.442767
max        3.000000
Name: 2026-08-04 00:00:00, dtype: float64


In [10]:
# Pick a random date and confirm the standardization worked as intended
check_date = momentum_standardized.dropna(how="all").index[-1]

print(f"Checking date: {check_date}")
print(f"Mean: {momentum_standardized.loc[check_date].mean():.4f} (should be close to 0)")
print(f"Std: {momentum_standardized.loc[check_date].std():.4f} (should be close to 1)")
print(f"Min: {momentum_standardized.loc[check_date].min():.4f}, Max: {momentum_standardized.loc[check_date].max():.4f} (should be within +/-3 after winsorizing)")

Checking date: 2026-08-04 00:00:00
Mean: -0.0224 (should be close to 0)
Std: 0.9066 (should be close to 1)
Min: -2.5092, Max: 3.0000 (should be within +/-3 after winsorizing)


In [11]:
mom_standardized_ic, mom_standardized_summary = evaluate_signal(momentum_standardized, close, horizon=HORIZON)
print("Momentum (standardized) IC summary:")
print(pd.Series(mom_standardized_summary).round(4))

print("\nMomentum (raw) IC summary, for comparison:")
print(pd.Series(mom_summary).round(4))

Momentum (standardized) IC summary:
Mean IC                -0.0147
IC Std                  0.1954
Information Ratio      -0.0754
% Positive IC           0.5126
N Observations       1824.0000
dtype: float64

Momentum (raw) IC summary, for comparison:
Mean IC                -0.0147
IC Std                  0.1954
Information Ratio      -0.0754
% Positive IC           0.5126
N Observations       1824.0000
dtype: float64


In [12]:
from modules.construction.simple_construction import quantile_weights, score_proportional_weights

In [13]:
momentum_quantile_weights = quantile_weights(momentum_standardized, long_pct=0.2, short_pct=0.2, gross_exposure=1.0)

check_date = momentum_quantile_weights[(momentum_quantile_weights != 0).any(axis=1)].index[-1]
print(f"Checking date: {check_date}")

nonzero_weights = momentum_quantile_weights.loc[check_date]
nonzero_weights = nonzero_weights[nonzero_weights != 0]

print(f"Number of positions: {len(nonzero_weights)}")
print(f"Sum of long weights: {nonzero_weights[nonzero_weights > 0].sum():.4f} (should be ~0.5)")
print(f"Sum of short weights: {nonzero_weights[nonzero_weights < 0].sum():.4f} (should be ~-0.5)")
print(f"Net exposure: {nonzero_weights.sum():.4f} (should be ~0, market-neutral by construction)")
print(f"Gross exposure: {nonzero_weights.abs().sum():.4f} (should be ~1.0)")

Checking date: 2026-08-04 00:00:00
Number of positions: 200
Sum of long weights: 0.5000 (should be ~0.5)
Sum of short weights: -0.5000 (should be ~-0.5)
Net exposure: 0.0000 (should be ~0, market-neutral by construction)
Gross exposure: 1.0000 (should be ~1.0)


In [14]:
momentum_proportional_weights = score_proportional_weights(momentum_standardized, gross_exposure=1.0)

nonzero_prop_weights = momentum_proportional_weights.loc[check_date]
nonzero_prop_weights = nonzero_prop_weights[nonzero_prop_weights != 0]

print(f"Number of positions: {len(nonzero_prop_weights)}")
print(f"Sum of long weights: {nonzero_prop_weights[nonzero_prop_weights > 0].sum():.4f} (should be ~0.5)")
print(f"Sum of short weights: {nonzero_prop_weights[nonzero_prop_weights < 0].sum():.4f} (should be ~-0.5)")
print(f"Net exposure: {nonzero_prop_weights.sum():.4f} (should be ~0)")
print(f"Gross exposure: {nonzero_prop_weights.abs().sum():.4f} (should be ~1.0)")

Number of positions: 501
Sum of long weights: 0.5000 (should be ~0.5)
Sum of short weights: -0.5000 (should be ~-0.5)
Net exposure: -0.0000 (should be ~0)
Gross exposure: 1.0000 (should be ~1.0)


In [15]:
mr_quantile_weights = quantile_weights(mean_reversion_standardized, long_pct=0.2, short_pct=0.2, gross_exposure=1.0)

nonzero_mr_weights = mr_quantile_weights.loc[check_date]
nonzero_mr_weights = nonzero_mr_weights[nonzero_mr_weights != 0]

print(f"Number of positions: {len(nonzero_mr_weights)}")
print(f"Net exposure: {nonzero_mr_weights.sum():.4f}")
print(f"Gross exposure: {nonzero_mr_weights.abs().sum():.4f}")

Number of positions: 200
Net exposure: -0.0000
Gross exposure: 1.0000


<div style="font-family: 'Helvetica Neue', -apple-system, Arial, sans-serif;">

### Dollar-Neutral vs. Beta-Neutral

*Dollar-neutral:* long dollar exposure equals short dollar exposure (net exposure ≈ 0). This is what construction module gives us automatically, as a side effect of the 50/50 long-short split.

*Beta-neutral:* the long side's *aggregate market sensitivity* equals the short side's, so the portfolio doesn't gain or lose purely from the overall market moving up or down. Dollar-neutrality does NOT guarantee this. If the longs happen to be higher-beta stocks than the shorts, the portfolio still has an unintended market-direction bet, even at zero net dollar exposure.

At this current moment, we're dollar-neutral only. Beta-neutrality (and sector-neutrality) hasn't been enforced yet — that's the job of the risk module next.
___

</div>

In [16]:
# Most recent date with actual positions
latest_date = momentum_quantile_weights[(momentum_quantile_weights != 0).any(axis=1)].index[-1]
latest_positions = momentum_quantile_weights.loc[latest_date]
latest_positions = latest_positions[latest_positions != 0].sort_values(ascending=False)

print(f"Positions as of: {latest_date}\n")

print("Top 5 Long:")
print(latest_positions.head(5))

print("\nTop 5 Short:")
print(latest_positions.tail(5))

Positions as of: 2026-08-04 00:00:00

Top 5 Long:
Ticker
A       0.005
HPQ     0.005
MET     0.005
LRCX    0.005
LH      0.005
Name: 2026-08-04 00:00:00, dtype: float64

Top 5 Short:
Ticker
KR     -0.005
CME    -0.005
CMS    -0.005
ISRG   -0.005
ZTS    -0.005
Name: 2026-08-04 00:00:00, dtype: float64


In [17]:
# Quick, informal preview only -- not the real allocation module yet,
# just a simple 50/50 blend to see what combining looks like
combined_weights = 0.5 * momentum_quantile_weights + 0.5 * mr_quantile_weights

latest_combined = combined_weights.loc[latest_date]
latest_combined = latest_combined[latest_combined != 0].sort_values(ascending=False)

print("Top 5 Long (combined, informal 50/50 blend):")
print(latest_combined.head(5))
print("\nTop 5 Short (combined, informal 50/50 blend):")
print(latest_combined.tail(5))

Top 5 Long (combined, informal 50/50 blend):
Ticker
DOC     0.005
CVS     0.005
CTAS    0.005
HUM     0.005
FDS     0.005
Name: 2026-08-04 00:00:00, dtype: float64

Top 5 Short (combined, informal 50/50 blend):
Ticker
BKR    -0.005
ORCL   -0.005
PWR    -0.005
SBAC   -0.005
BSX    -0.005
Name: 2026-08-04 00:00:00, dtype: float64


<div style="font-family: 'Helvetica Neue', -apple-system, Arial, sans-serif;">

### Construction vs. Allocation (Important Distinction)

"Construction" builds each signal's own strategy independently: Momentum's positions, Mean Reversion's positions, each fully formed on its own. No combining happens here.

Combining strategies happens later, under "Allocation," after each strategy has passed through its own risk checks (via the risk and limit modules) individually. Allocation decides how much capital each *already-built* strategy gets, then sums them into one final book.

We're still upstream of that — "momentum_quantile_weights" and "mr_quantile_weights" are two separate, uncombined strategies, exactly as intended at this stage.
___

</div>

In [18]:
from modules.signals.fetch import fetch_price_data

spy_close, spy_volume = fetch_price_data(["SPY"], start=START_DATE, end=END_DATE)

# Merge SPY into the existing close DataFrame so beta can be computed
# directly from data we already have, per the self-contained design
close_with_spy = close.join(spy_close)
close_with_spy.tail()

Fetching batch 1 / 1 (1 tickers)...


[*********************100%***********************]  1 of 1 completed


Ticker,A,AAPL,ABBV,ABNB,ABT,ACN,ADBE,ADI,AEE,AEP,...,WTW,WY,WYNN,XEL,XYL,YUM,ZBH,ZBRA,ZTS,SPY
Date,,,,,,,,,,,,,,,,,,,,,
2026-07-29,140.300003,337.898590,263.299988,153.009995,108.000000,173.169998,263.429993,353.369995,109.970001,128.422256,...,315.929993,24.450001,98.768211,78.750000,122.129997,151.919998,97.150002,282.980011,77.930000,729.460022
2026-07-30,138.710007,333.142670,257.410004,152.080002,105.610001,163.289993,247.899994,366.670013,108.750000,126.814507,...,336.049988,23.500000,100.454079,78.230003,116.820000,157.000000,94.959999,288.570007,76.029999,741.690002
2026-07-31,138.369995,308.643829,250.940002,151.520004,105.699997,165.919998,250.410004,367.410004,NaN,126.883980,...,NaN,25.030001,NaN,78.199997,116.970001,153.279999,93.930000,NaN,77.290001,747.030029
2026-08-03,139.800003,303.158569,245.100006,150.639999,107.120003,165.759995,251.339996,361.880005,109.500000,127.350433,...,341.630005,25.230000,98.119797,77.690002,119.080002,148.800003,96.980003,291.640015,77.099998,757.669983
2026-08-04,139.199997,309.113403,243.800003,149.919998,105.459999,170.429993,257.489990,380.290009,109.440002,127.380203,...,339.630005,25.830000,97.361649,77.750000,122.169998,147.679993,95.809998,368.829987,76.040001,771.330017


In [19]:
from modules.risk.factor_exposure import compute_rolling_beta

betas = compute_rolling_beta(close_with_spy, market_ticker="SPY", window=252)

check_date = betas.dropna(how="all").index[-1]
print(f"Checking date: {check_date}")
betas.loc[check_date].describe()

Checking date: 2026-08-04 00:00:00


count    379.000000
mean       0.745768
std        0.857193
min       -0.852952
25%        0.121623
50%        0.602779
75%        1.172284
max        4.332649
Name: 2026-08-04 00:00:00, dtype: float64

In [20]:
from modules.risk.factor_exposure import compute_portfolio_factor_exposure

momentum_beta_exposure = compute_portfolio_factor_exposure(momentum_quantile_weights, betas)

print(momentum_beta_exposure.dropna().tail(10))
print(f"\nMean beta exposure: {momentum_beta_exposure.mean():.4f}")

Date
2026-07-22    0.204088
2026-07-23    0.088472
2026-07-24    0.005901
2026-07-27   -0.076114
2026-07-28   -0.128482
2026-07-29   -0.154958
2026-07-30    0.039204
2026-07-31   -0.058301
2026-08-03   -0.039620
2026-08-04    0.107609
dtype: float64

Mean beta exposure: 0.0359


In [21]:
from modules.risk.factor_exposure import compute_sector_exposure

momentum_sector_exposure = compute_sector_exposure(momentum_quantile_weights, universe)

momentum_sector_exposure.loc[check_date].sort_values()

Communication Services   -0.065
Utilities                -0.060
Energy                   -0.050
Consumer Staples         -0.010
Materials                -0.010
Consumer Discretionary   -0.005
Industrials               0.005
Real Estate               0.005
Financials                0.050
Information Technology    0.050
Health Care               0.090
Name: 2026-08-04 00:00:00, dtype: float64

<div style="font-family: 'Helvetica Neue', -apple-system, Arial, sans-serif;">

### Confirmation: Dollar-Neutral does not equal Factor-Neutral

The Momentum portfolio is dollar-neutral by construction (net exposure ~ 0), but the measurements above show it carries unintended factor bets:

- *Beta exposure* swings between roughly -0.15 and +0.20 day to day, with a mean around 0.036, a persistent tilt toward benefiting when the broad market rallies, despite no such bet being intended.
- *Sector exposure* shows Health Care at +9%, Information Technology and Financials both at +5%, versus Communication Services at -6.5%, Utilities at -6.0%, and Energy at -5.0%. This reveals what's effectively a  sector rotation bet riding along inside what's supposed to be a pure stock-picking signal.

This confirms precisely why we need the "risk" module: dollar-neutrality alone doesn't stop a portfolio from accidentally betting on the market or on specific sectors.
___

</div>

In [ ]:
from modules.risk.neutralization import sector_neutralize, beta_neutralize

momentum_sector_neutral = sector_neutralize(momentum_quantile_weights, universe)
momentum_sector_neutral_exposure = compute_sector_exposure(momentum_sector_neutral, universe)

print("Sector exposure AFTER naive neutralization:")
momentum_sector_neutral_exposure.loc[check_date].sort_values()

In [ ]:
momentum_beta_neutral = beta_neutralize(momentum_quantile_weights, betas)
momentum_beta_neutral_exposure = compute_portfolio_factor_exposure(momentum_beta_neutral, betas)

print("Beta exposure AFTER naive neutralization:")
print(momentum_beta_neutral_exposure.dropna().tail(10))
print(f"\nMean beta exposure after: {momentum_beta_neutral_exposure.mean():.4f}")